<div>
<center><img src="../assets/Flux-logo.svg" width="360"/></center>
</div>

<div style="background:linear-gradient(90deg,#036291 0%,#91C2D8 100%);padding:20px 26px;border-radius:10px;border-left:10px solid #D9A441;margin-top:18px">
<h1 style="margin:0;color:#ffffff">Module 1: The Flux Python SDK 🐍️</h1>
<p style="margin:6px 0 0 0;color:#DCECF4;font-size:15px">Submitting and watching jobs programmatically</p>
<p style="margin:2px 0 0 0;color:#DCECF4;font-size:13px">SC26 &middot; Chicago &middot; November 2026</p>
</div>

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Run these:</span> This is the one notebook with executable cells. Everything else is copy-paste into a terminal, because commands like <code>flux top</code> and <code>flux alloc</code> need a real TTY.
</div>

## The handle

Flux needs Python to build, so if Flux is installed its Python bindings are too. (For a
separate interpreter, `pip install flux-python`.) Everything starts from a handle, which is
a connection to the instance named by `FLUX_URI`:

In [ ]:
import json
import os

import flux
import flux.job
from flux.job import JobspecV1, JobID

handle = flux.Flux()
print(handle)

## Jobspec

<div class="alert alert-block" style="background-color:#036291;color:#ffffff">
<span style="font-weight:600"><code>JobspecV1</code> builds a job description from a command and resource counts</span>
</div>

Flux describes work as a [jobspec](https://flux-framework.readthedocs.io/projects/flux-rfc/en/latest/spec_25.html).
You could write the YAML by hand, but `JobspecV1.from_command` builds one from a command
and resource counts, and everything else is an attribute you set on the object.

In [ ]:
jobspec = JobspecV1.from_command(
    command=["sleep", "5"], num_tasks=1, num_nodes=1, cores_per_task=1
)
jobspec.environment = {"TMPDIR": "/tmp"}

jobid = flux.job.submit(handle, jobspec)
print(f"{jobid.f58 = }")

`submit` blocks only until the job has an ID. Ask about it right away with `get_job`:

In [ ]:
info = flux.job.get_job(handle, jobid)
print(info["status"], info["t_run"])

The status is probably `SCHED` or `RUN` and `t_run` is zero or very recent: the job has
not finished. `flux.job.result` waits for it and returns the outcome.

In [ ]:
result = flux.job.result(handle, jobid)
print(json.dumps(result.to_dict(), indent=2))

### One job, several names

A `JobID` is an integer with several encodings. The `ƒ` form (`f58`) is what the CLI
prints. `words` and `emoji` are real, and occasionally useful for reading an ID aloud.

In [ ]:
fid = JobID(jobid)
for form in ("dec", "hex", "f58", "words", "emoji"):
    print(f"{form:>6}: {getattr(fid, form)}")

The CLI sees the same jobs. A leading `!` runs a shell command from a notebook cell:

In [ ]:
!flux jobs -a --name=sleep

Under the hood the jobspec is JSON. This is exactly what was sent to Flux:

In [ ]:
print(jobspec.dumps(indent=2))

### CLI flags and their Python equivalents

Some jobspec attributes are arguments to `from_command`; the rest are set on the object
afterward.

| Operation | CLI | Python |
|---|---|---|
| Nodes, tasks, cores | `-N`, `-n`, `-c` | `from_command(num_nodes=, num_tasks=, cores_per_task=)` |
| Node exclusivity | `-x` | `from_command(exclusive=True)` or `jobspec.exclusive = True` |
| Time limit | `-t 5m` | `jobspec.duration = "5m"` |
| Working directory | `--cwd=` | `jobspec.cwd = path` |
| Environment | `--env=` | `jobspec.environment = dict` |
| Output files | `--output`, `--error` | `jobspec.stdout`, `jobspec.stderr` |
| System attributes | `-S KEY=VAL` | `jobspec.setattr(key, val)` |
| Shell options | `-o KEY=VAL` | `jobspec.setattr_shell_option(key, val)` |
| Sub-instance config | `--conf` (batch, alloc) | `from_batch_command`, `from_nest_command` |

This example is adapted from the
[El Capitan documentation](https://hpc.llnl.gov/documentation/user-guides/using-el-capitan-systems/introduction-and-quickstart/flux)
maintained by Ramesh Pankajakshan. `event_wait` blocks until a named event appears in the
job's eventlog.

In [ ]:
jobspec = JobspecV1.from_command(command=["sleep", "2"], num_tasks=4, num_nodes=1)
jobspec.cwd = os.getcwd()
jobspec.exclusive = True
jobspec.duration = "5m"
jobspec.environment = {"TMPDIR": "/tmp"}
jobspec.stdout = "/tmp/pyexample.{{id}}.out"
jobspec.stderr = "/tmp/pyexample.{{id}}.err"

jobid = flux.job.submit(handle, jobspec)
print(flux.job.event_wait(handle, jobid, "finish"))

## Many jobs: FluxExecutor

<div class="alert alert-block" style="background-color:#036291;color:#ffffff">
<span style="font-weight:600"><code>FluxExecutor</code> follows the <code>concurrent.futures</code> interface</span>
</div>

`submit` returns a future, and `as_completed` yields futures as their jobs finish. This is
how you submit thousands of jobs from one process.

In [ ]:
import concurrent.futures
from flux.job import FluxExecutor

jobspec = JobspecV1.from_command(["hostname"])
jobspec.environment = {"TMPDIR": "/tmp"}

with FluxExecutor() as executor:
    futures = [executor.submit(jobspec) for _ in range(5)]
    for fut in concurrent.futures.as_completed(futures):
        print(fut.jobid(), "exit status", fut.result())

[bulksubmit_executor.py](bulksubmit_executor.py) wraps the same loop in a progress bar and
reports throughput. Try more jobs, or race it against
`flux submit --cc=1-200 --watch --progress --jps true` in the terminal.

In [ ]:
!flux python bulksubmit_executor.py -n200 /bin/true

## Getting output

<div class="alert alert-block" style="background-color:#036291;color:#ffffff">
<span style="font-weight:600">Three ways to read a job's output, in increasing order of control</span>
</div>

`job_output` waits for the job to finish and returns everything. `output_watch_lines` is a
generator that yields `(stream, line)` pairs as output arrives. `event_watch` hands you the
raw eventlog entries that both of those are built on.

In [ ]:
import flux.job.output

joke = "Why did the Cubs fan love Flux? Finally, something that waits less than 108 years."
jobspec = JobspecV1.from_command(["echo", joke])
jobspec.environment = {"TMPDIR": "/tmp"}
jobid = flux.job.submit(handle, jobspec)

print(flux.job.output.job_output(handle, jobid).stdout)

In [ ]:
joke = "The Chicago River flows backwards. It took one look at the job queue and turned around."
jobspec = JobspecV1.from_command(["echo", joke], num_tasks=3)
jobspec.environment = {"TMPDIR": "/tmp"}
jobid = flux.job.submit(handle, jobspec)

for stream, line in flux.job.output.output_watch_lines(handle, jobid):
    print(f"{stream}: {line}")

In [ ]:
for event in flux.job.event_watch(handle, jobid, "guest.output"):
    print(event.name, event.context)

With an executor you do not have to wait at all. Attach a callback to each future and it
runs when the job finishes:

In [ ]:
def print_output(fut):
    out = flux.job.output.job_output(handle, fut.jobid()).stdout
    print(f"{fut.jobid()}: {out.strip()}")

jobspec = JobspecV1.from_command(["echo", "Dibs! 🪑 This core is mine until the snow melts."])
jobspec.environment = {"TMPDIR": "/tmp"}

with FluxExecutor() as executor:
    for _ in range(3):
        executor.submit(jobspec).add_done_callback(print_output)

## Listing jobs

<div class="alert alert-block" style="background-color:#036291;color:#ffffff">
<span style="font-weight:600"><code>JobList</code> is the query behind <code>flux jobs</code></span>
</div>

`JobList(...).jobs()` sends a `job-list` RPC to the job manager and returns one `JobInfo`
per job, with attributes like `status` and `runtime`. The timestamps are plain floats you
can do arithmetic on.

In [ ]:
from flux.job.list import JobList

jobs = JobList(handle).jobs()
for job in jobs[:8]:
    waited = job.t_run - job.t_submit if job.t_run else float("nan")
    print(f"{job.name:>12} {job.status:>10} waited {waited:6.3f}s, ran {job.runtime:6.3f}s")

## The KVS, and the reactor

<div class="alert alert-block" style="background-color:#036291;color:#ffffff">
<span style="font-weight:600">The key-value store underpins most of Flux's own services</span>
</div>

The Python bindings expose the KVS directly. This example also shows the event loop
everything in Flux runs on: a timer fires after six seconds, writes a menu into the KVS,
reads it back, and fulfills a future whose callback stops the reactor. The deep dish, as
always, takes a while.

In [ ]:
from functools import partial

import flux.future
import flux.kvs


def order_placed(future):
    print("Order placed. The deep dish takes a while. Nobody is sorry.")
    with flux.kvs.KVSTxn(flux_handle=handle) as txn:
        txn.mkdir("menu")
        txn.put("menu.deep_dish", "45 minutes, worth it")
        txn.put("menu.italian_beef", "dipped, with hot giardiniera")
        txn.put("menu.hot_dog", "dragged through the garden, no ketchup")


def order_up(h, watcher, revents, args, future):
    menu = flux.kvs.KVSDir(h, "menu")
    for item in menu.files():
        print(f"  {item}: {flux.kvs.get(h, menu.key_at(item))}")
    future.fulfill()


def table_served(future):
    print("Order up. Stopping the reactor.")
    handle.reactor_stop()


order = flux.future.FutureExt(order_placed, flux_handle=handle)
order.then(table_served)

timer = handle.timer_watcher_create(6, partial(order_up, future=order))
timer.start()
handle.reactor_run()

The KVS is one store shared by every rank and every language. In the terminal,
`flux kvs get menu.hot_dog` returns the value you just wrote from Python.

<br>

## 🧭 Side quests

<div class="alert alert-block" style="background-color:#D9A441;color:#06293D">
<span style="font-weight:600">Finished early?</span> Pick one. Each takes about ten minutes, and an answer is folded up underneath it. Try before you unfold.
</div>

### 1. Who waited longest?

Using `JobList`, find the job in this instance that waited longest between submit and
run, and print its ID, name, and the wait. Then find the one that ran longest. Watch out
for jobs that never ran.

<details><summary>One answer</summary>

```python
jobs = JobList(handle, max_entries=5000).jobs()
ran = [j for j in jobs if j.t_run]

slow = max(ran, key=lambda j: j.t_run - j.t_submit)
print(slow.id.f58, slow.name, f"waited {slow.t_run - slow.t_submit:.2f}s")

longest = max(ran, key=lambda j: j.runtime)
print(longest.id.f58, longest.name, f"ran {longest.runtime:.1f}s")
```

If you did the six-core oven in notebook 1, the pizzas that waited are the answer to
"does order matter" in numbers.
</details>

### 2. The hot dog, in Python

Notebook 1 built a Chicago-style hot dog by chaining `bulksubmit/0.sh` through `7.sh` with
`--dependency=afterok`. Do it from Python: eight jobspecs, each depending on the previous
job, then print the output in order.

Hint: dependencies live in the jobspec at `system.dependencies` as a list of
`{"scheme": "afterok", "value": "<jobid>"}` dicts, and `jobspec.setattr` sets them.

<details><summary>One answer</summary>

```python
prev, ids = None, []
for i in range(8):
    js = JobspecV1.from_command(["bash", f"bulksubmit/{i}.sh"])
    js.environment = {"TMPDIR": "/tmp"}
    if prev is not None:
        js.setattr("system.dependencies", [{"scheme": "afterok", "value": str(prev)}])
    prev = flux.job.submit(handle, js)
    ids.append(prev)

for jid in ids:
    print(flux.job.output.job_output(handle, jid).stdout, end="")
```

Follow-up: `flux.job.get_job(handle, ids[-1])["dependencies"]` shows what the last job is
waiting on, right up until it isn't.
</details>

### 3. Leave a note for the terminal

Write a key into the KVS from Python and read it from the terminal with `flux kvs get`.
Then go the other way: `flux kvs put` in the terminal, read it back here. Small, but it is
the mechanism `flux archive` in the next notebook is built on.

<details><summary>One answer</summary>

```python
flux.kvs.put(handle, "notes.from_python", "the oven is preheated")
flux.kvs.commit(handle)
```

```bash
flux kvs get notes.from_python
flux kvs put notes.from_terminal="no ketchup"
```

```python
print(flux.kvs.get(handle, "notes.from_terminal"))
```
</details>

<div style="background:#DCECF4;border-left:6px solid #D9A441;padding:12px 18px;color:#06293D"><strong>Module 1, Notebook 2 complete</strong></div>

Next: [instances, internals, and plugins](03_flux_instances_and_plugins.ipynb).